In [1]:
import numpy as np
import matplotlib.pyplot as plt

In [ ]:
# -----------------------------
# Part A: Phase-reduced model
# -----------------------------
def arnold_tongue_phase_model(delta_min=-0.6, delta_max=0.6, K_min=0.0, K_max=0.6, n_delta=201, n_K=201):
    """
    Mutual (symmetric) sine-coupling phase model:
      theta1' = w1 + K sin(theta2 - theta1)
      theta2' = w2 + K sin(theta1 - theta2)
    Relative phase phi = theta2 - theta1 obeys: phi' = Δω - 2K sin(phi)
    Locking exists iff |Δω| <= 2K. This yields a V-shaped 1:1 Arnold tongue.
    """
    deltas = np.linspace(delta_min, delta_max, n_delta)
    Ks = np.linspace(K_min, K_max, n_K)
    locked = np.zeros((n_K, n_delta), dtype=bool)

    for i, K in enumerate(Ks):
        locked[i, :] = np.abs(deltas) <= 2.0 * K

    D, Kgrid = np.meshgrid(deltas, Ks)
    plt.figure(figsize=(6, 5))
    plt.contourf(D, Kgrid, locked.astype(float), levels=[-0.5, 0.5, 1.5])
    plt.xlabel(r"detuning $\Delta\omega = \omega_2-\omega_1$")
    plt.ylabel("coupling K")
    plt.title("Arnold tongue (1:1) — phase model (|Δω| ≤ 2K)")
    plt.plot([delta_min, delta_max], [0.0, 0.0], linewidth=1)  # axis line
    plt.show()

In [ ]:
# --------------------------------------
# Part B: Full two–van der Pol oscillators
# --------------------------------------
def vdp_coupled_rhs(y, t, mu, w1, w2, k):
    """
    y = [x1, v1, x2, v2]
    x'' - mu(1 - x^2)x' + w^2 x = coupling
    We write as first order:
      x' = v
      v' = mu(1 - x^2)*v - w^2*x + k*(x_other - x)
    """
    x1, v1, x2, v2 = y
    dx1 = v1
    dv1 = mu*(1.0 - x1**2)*v1 - (w1**2)*x1 + k*(x2 - x1)
    dx2 = v2
    dv2 = mu*(1.0 - x2**2)*v2 - (w2**2)*x2 + k*(x1 - x2)
    return np.array([dx1, dv1, dx2, dv2])

def rk4_step(f, y, t, dt, *args):
    k1 = f(y, t, *args)
    k2 = f(y + 0.5*dt*k1, t + 0.5*dt, *args)
    k3 = f(y + 0.5*dt*k2, t + 0.5*dt, *args)
    k4 = f(y + dt*k3, t + dt, *args)
    return y + (dt/6.0)*(k1 + 2*k2 + 2*k3 + k4)

def simulate_two_vdp(mu, w1, w2, k, dt=0.02, T=500.0, T_transient=200.0, y0=None):
    """
    Integrate two coupled VdP oscillators.
    Returns: time, x1, x2
    """
    n = int(np.round(T/dt))
    t = np.linspace(0.0, T, n+1)
    if y0 is None:
        # Mildly different initials to avoid accidental symmetry
        y = np.array([2.0, 0.0, 0.1, 0.0], dtype=float)
    else:
        y = np.array(y0, dtype=float)

    X1 = np.empty(n+1); X2 = np.empty(n+1)
    X1[0] = y[0]; X2[0] = y[2]

    for i in range(n):
        y = rk4_step(vdp_coupled_rhs, y, t[i], dt, mu, w1, w2, k)
        X1[i+1] = y[0]; X2[i+1] = y[2]

    # Trim transient
    idx0 = np.searchsorted(t, T_transient)
    return t[idx0:], X1[idx0:], X2[idx0:]

def mean_frequency_from_zerocrossings(t, x):
    """
    Estimate average frequency via positive-going zero crossings.
    Returns np.nan if too few cycles.
    """
    x = np.asarray(x)
    t = np.asarray(t)
    # Detect zero-crossings with positive slope
    s = np.signbit(x)  # True where x < 0
    crossings = np.where((s[:-1] == True) & (s[1:] == False) & (x[1:] > x[:-1]))[0]
    if crossings.size < 5:
        return np.nan
    # Linear interpolation for crossing times
    tc = []
    for i in crossings:
        x0, x1 = x[i], x[i+1]
        if x1 == x0:
            continue
        frac = -x0 / (x1 - x0)
        tc.append(t[i] + (t[i+1]-t[i]) * frac)
    tc = np.array(tc)
    if tc.size < 5:
        return np.nan
    periods = np.diff(tc)
    if periods.size < 3:
        return np.nan
    return 1.0 / np.mean(periods)

def arnold_tongue_vdp(mu=0.3, w1=1.0,
                      delta_min=-0.5, delta_max=0.5, n_delta=21,
                      k_min=0.0, k_max=0.5, n_k=15,
                      dt=0.02, T=400.0, T_transient=200.0,
                      freq_tol=0.01):
    """
    Compute a 1:1 locking diagram for two coupled VdP oscillators.
    Locking criterion: |f1 - f2| <= freq_tol (absolute Hz/rad/s scale),
    with both frequencies successfully estimated.
    """
    deltas = np.linspace(delta_min, delta_max, n_delta)
    Ks = np.linspace(k_min, k_max, n_k)

    locked = np.zeros((n_k, n_delta), dtype=bool)
    f1_grid = np.full((n_k, n_delta), np.nan)
    f2_grid = np.full((n_k, n_delta), np.nan)

    # Reuse last state as warm start across parameter scans (optional improvement)